**LeNet-5**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.


Dataset Preparation

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

# Global transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
]) ## This section written by gemini

def get_dataloaders(split_name, batch_size=64):
    train_dataset = torchvision.datasets.EMNIST(
        root='./data', train=True, split=split_name, download=True, transform=transform
    )
    test_dataset = torchvision.datasets.EMNIST(
        root='./data', train=False, split=split_name, download=True, transform=transform
    )
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Determine number of classes from the dataset labels
    num_classes = len(train_dataset.classes)
    return train_loader, test_loader, num_classes


Custom Compenents

In [2]:
import torch

def one_hot_encode(label, num_classes):
    """
    Converts an integer label into a one-hot encoded PyTorch tensor.

    Args:
        label (int): The integer class label.
        num_classes (int): The total number of classes.

    Returns:
        torch.Tensor: A one-hot encoded tensor of shape (num_classes,).
    """
    one_hot = torch.zeros(num_classes)
    one_hot[label] = 1.0
    return one_hot

Custom RBF Loss



The RBF layer computes the squared Euclidean distance between the input feature vector and a set of learnable class prototypes.

In [3]:
import torch
import torch.nn as nn

class RBFLayer(nn.Module):
    def __init__(self, in_features, num_classes):
        super(RBFLayer, self).__init__()
        self.in_features = in_features
        self.num_classes = num_classes

        # Initialize prototypes as learnable parameters from a normal distribution
        self.prototypes = nn.Parameter(torch.randn(num_classes, in_features)) # written by gemini

    def forward(self, x):
        # Calculate squared Euclidean distance: ||x - prototype||^2
        # Sum across feature dimensions
        distances = torch.sum((x.unsqueeze(1) - self.prototypes.unsqueeze(0))**2, dim=2) # written by gemini

        return distances

print("RBFLayer class defined successfully.")

RBFLayer class defined successfully.


Final Model Definition

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet5(nn.Module):
    def __init__(self, num_classes):
        super(LeNet5, self).__init__()
        # C1: Convolutional layer (1 input channel, 6 output channels, 5x5 kernel, padding=2)
        # Input size: 32x32, Output size: (32 - 5 + 2*2)/1 + 1 = 32x32
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        # S2: Average pooling (2x2 kernel, stride=2)
        # Input size: 32x32, Output size: (32 - 2)/2 + 1 = 16x16

        # C3: Convolutional layer (6 input channels, 16 output channels, 5x5 kernel)
        # Input size: 16x16, Output size: (16 - 5)/1 + 1 = 12x12
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        # S4: Average pooling (2x2 kernel, stride=2)
        # Input size: 12x12, Output size: (12 - 2)/2 + 1 = 6x6

        # F5: Fully connected layer (16*6*6 = 576 input features, 120 output features)
        self.fc1 = nn.Linear(16 * 6 * 6, 120)
        # F6: Fully connected layer (120 input features, 84 output features)
        self.fc2 = nn.Linear(120, 84)

        # The output of fc2 (F6 layer) is 84 features, which becomes the in_features for RBF.
        self.rbf_layer = RBFLayer(84, num_classes)

    def forward(self, x):
        # C1 -> Tanh -> S2 (AvgPool) -> Tanh
        x = F.tanh(self.conv1(x))
        x = F.tanh(F.avg_pool2d(x, kernel_size=2, stride=2))

        # C3 -> Tanh -> S4 (AvgPool) -> Tanh
        x = F.tanh(self.conv2(x))
        x = F.tanh(F.avg_pool2d(x, kernel_size=2, stride=2))

        # Flatten for fully connected layers
        x = x.view(-1, 16 * 6 * 6)

        # F5 -> Tanh
        x = F.tanh(self.fc1(x))

        # F6 -> Tanh
        x = F.tanh(self.fc2(x))

        # Output layer is the RBF layer
        x = self.rbf_layer(x) # Custom RBF layer
        return x

In [9]:
import torch
import torch.nn as nn

def custom_rbf_loss(distances, labels, margin=1.0):
    batch_size = distances.size(0)
    num_classes = distances.size(1)

    # Get the distances to the correct prototypes
    distances_to_correct_prototype = distances.gather(1, labels.unsqueeze(1)).squeeze(1) # written by gemini

    # Create a mask for incorrect prototypes
    incorrect_mask = torch.ones_like(distances, dtype=torch.bool)
    incorrect_mask.scatter_(1, labels.unsqueeze(1), 0) ## written by gemini

    # Get the distances to all incorrect prototypes
    distances_to_incorrect_prototypes = distances[incorrect_mask].view(batch_size, num_classes - 1) ## written by gemini

    # Term 1: Minimize distance to correct prototype
    loss_correct = distances_to_correct_prototype.mean()

    # Term 2: Maximize distance to incorrect prototypes via margin penalty
    if distances_to_incorrect_prototypes.numel() > 0:
        loss_incorrect = torch.clamp(margin - distances_to_incorrect_prototypes, min=0).mean() # written by gemini
    else:
        loss_incorrect = torch.tensor(0.0, device=distances.device)

    total_loss = loss_correct + loss_incorrect

    return total_loss


Custom RBF loss function `custom_rbf_loss` defined successfully.


In [10]:
import torch.optim as optim
import matplotlib.pyplot as plt
import os

def train_evaluate_save(split_name, epochs=30):
    print(f"\n--- Processing EMNIST split: {split_name} ---")
    save_base_dir = '/content/drive/MyDrive/cs320/final project/models'
    os.makedirs(save_base_dir, exist_ok=True) # written by gemini

    # 1. Prepare Data
    train_loader, test_loader, num_classes = get_dataloaders(split_name)

    # 2. Initialize Model, Optimizer, and Loss
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LeNet5(num_classes).to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = custom_rbf_loss

    # 3. Training Loop
    for epoch in range(1, epochs + 1): # Start epoch from 1 for saving logic
        model.train()
        total_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"[{split_name}] Epoch {epoch} complete. Avg Loss: {total_loss/len(train_loader):.4f}")

        # Save model at specific epochs to Google Drive
        if epoch == 5 or epoch == 10 or epoch == 20:
            save_path_epoch = os.path.join(save_base_dir, f"lenet5_rbf_{split_name}_epoch_{epoch}.pth")
            torch.save(model.state_dict(), save_path_epoch)
            print(f"Model saved to {save_path_epoch} at epoch {epoch}")
            ## Saving section written by gemini

    # 4. Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            # In RBF, the smallest distance indicates the predicted class
            _, predicted = torch.min(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Accuracy on {split_name}: {accuracy:.2f}%")

    # Save models
    final_save_path = os.path.join(save_base_dir, f"lenet5_rbf_{split_name}_final.pth")
    print(f"Final model saved to {final_save_path}")
    return accuracy

# Execute for each split
splits = ['bymerge','byclass', 'balanced']
results = {}

for s in splits:
    results[s] = train_evaluate_save(s, epochs=5)

print("\nSummary of Results:")
for split, acc in results.items():
    print(f"{split}: {acc:.2f}%")


--- Processing EMNIST split: byclass ---


100%|██████████| 562M/562M [00:03<00:00, 184MB/s]


KeyboardInterrupt: 

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def evaluate_model_comprehensive(model, test_loader, device, split_name):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            # In RBF, the smallest distance indicates the predicted class
            _, predicted = torch.min(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    print(f"Metrics for {split_name}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}\n")
    ## Gemini wrote this print formatting

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

In [14]:
save_base_dir = '/content/drive/MyDrive/cs320/final project/models'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
splits_to_eval = ['bymerge','byclass', 'balanced']
final_results = {}

for split in splits_to_eval:
    print(f"--- Evaluating final model for split: {split} ---")
    train_loader, test_loader, num_classes = get_dataloaders(split)

    # Initialize model and load weights
    model = LeNet5(num_classes).to(device)
    model_path = os.path.join(save_base_dir, f"lenet5_rbf_{split}_final.pth")

    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        final_results[split] = evaluate_model_comprehensive(model, test_loader, device, split)
    else:
        print(f"Model file not found at {model_path}")

--- Evaluating final model for split: bymerge ---
Metrics for bymerge:
  Accuracy:  0.7864
  Precision: 0.8047
  Recall:    0.7864
  F1 Score:  0.7832

--- Evaluating final model for split: byclass ---
Metrics for byclass:
  Accuracy:  0.7333
  Precision: 0.7593
  Recall:    0.7333
  F1 Score:  0.7303

--- Evaluating final model for split: balanced ---
Metrics for balanced:
  Accuracy:  0.8066
  Precision: 0.8079
  Recall:    0.8066
  F1 Score:  0.8045



Custom Dataset


In [5]:
import os
import torch
from PIL import Image, ImageOps
import torchvision.transforms as transforms

# --- 1. Label Mappings ---

# Map your custom file numbers to the actual lowercase characters
custom_to_char = {
    1: 'q', 2: 'l', 3: 'p', 4: 'o', 5: 'w', 6: 't', 7: 'v', 8: 'n',
    9: 'h', 10: 'g', 11: 'f', 12: 'd', 13: 'k', 14: 'a', 15: 'c',
    16: 'u', 17: 'j', 18: 'x', 19: 'i', 20: 's', 21: 'e', 22: 'r',
    23: 'b', 24: 'z', 25: 'y', 26: 'm'
}

# Map characters to EMNIST 'byclass' indices (lowercase 'a' is 36, 'z' is 61)
char_to_emnist_byclass = {chr(i + 97): i + 36 for i in range(26)}

# --- 2. Preprocessing Function ---
def preprocess_custom_image(image_path, is_dark_text=True, fix_emnist_rotation=True):
    img = Image.open(image_path).convert('L')

    if is_dark_text:
        img = ImageOps.invert(img)

    img = img.resize((32, 32), Image.Resampling.BILINEAR)

    if fix_emnist_rotation:
        img = img.rotate(-90, expand=True)
        img = ImageOps.mirror(img)

    transform_pipeline = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    return transform_pipeline(img).unsqueeze(0)

# --- 3. Evaluation Loop ---
def evaluate_custom_dataset(model, folder_path, device):
    model.eval() # Set model to evaluation mode

    results = {
        'total': {'correct': 0, 'count': 0},
        'messy': {'correct': 0, 'count': 0},
        'print': {'correct': 0, 'count': 0}
    }

    print(f"Evaluating images in: {folder_path}...\n")

    with torch.no_grad():
        for filename in os.listdir(folder_path):
            if not filename.endswith(('.png', '.jpg', '.jpeg')):
                continue

            # Parse filename (e.g., "sample1_messy_14.png")
            parts = filename.replace('.png', '').replace('.jpg', '').split('_')
            if len(parts) != 3:
                continue

            writing_type = parts[1]        # 'messy' or 'print'
            custom_class = int(parts[2])   # e.g., 14

            # Translate custom class to EMNIST class
            char = custom_to_char[custom_class]
            true_emnist_label = char_to_emnist_byclass[char]

            # Load and predict
            img_path = os.path.join(folder_path, filename)
            img_tensor = preprocess_custom_image(img_path).to(device)

            outputs = model(img_tensor)
            _, predicted_label = torch.max(outputs.data, 1)
            predicted_label = predicted_label.item()

            # Check if correct
            is_correct = (predicted_label == true_emnist_label)

            # Update stats
            results['total']['count'] += 1
            results[writing_type]['count'] += 1
            if is_correct:
                results['total']['correct'] += 1
                results[writing_type]['correct'] += 1

            # Optional: Print out the failures to see what the model struggles with
            if not is_correct:
                pred_char = [k for k, v in char_to_emnist_byclass.items() if v == predicted_label]
                pred_char = pred_char[0] if pred_char else f"Class {predicted_label}"
                print(f"Mistake on {filename}: True={char}, Predicted={pred_char}")

    # --- Print Summary ---
    print("\n--- Custom Dataset Results ---")
    for category in ['total', 'messy', 'print']:
        count = results[category]['count']
        if count > 0:
            acc = results[category]['correct'] / count * 100
            print(f"{category.capitalize()} Accuracy: {acc:.2f}% ({results[category]['correct']}/{count})")
        else:
            print(f"{category.capitalize()} Accuracy: N/A (0 samples)")


Custom Dataset

In [8]:
import os
import torch
from PIL import Image, ImageOps
import torchvision.transforms as transforms

# --- 1. Label Mappings ---

# Map your custom file numbers to the actual lowercase characters
custom_to_char = {
    1: 'q', 2: 'l', 3: 'p', 4: 'o', 5: 'w', 6: 't', 7: 'v', 8: 'n',
    9: 'h', 10: 'g', 11: 'f', 12: 'd', 13: 'k', 14: 'a', 15: 'c',
    16: 'u', 17: 'j', 18: 'x', 19: 'i', 20: 's', 21: 'e', 22: 'r',
    23: 'b', 24: 'z', 25: 'y', 26: 'm'
}

# Map characters to EMNIST 'byclass' indices (lowercase 'a' is 36, 'z' is 61)
char_to_emnist_byclass = {chr(i + 97): i + 36 for i in range(26)}

def preprocess_custom_image(image_path, is_dark_text=True, fix_emnist_rotation=True):
    # 1. Load image in grayscale
    img = Image.open(image_path).convert('L')

    # 2. Invert so text is white on black background
    if is_dark_text:
        img = ImageOps.invert(img)

    # 3. THRESHOLDING: Force dark greys (shadows/paper) to pure black and text to pure white
    # Anything below the pixel value of 100 becomes 0. You can tweak this number (e.g., 80 to 150)
    # if your lighting was particularly bright or dark!
    img = img.point(lambda p: 255 if p > 100 else 0)

    # 4. AUTO-CROPPING: Remove empty black space so the letter fills the frame like EMNIST
    bbox = img.getbbox() # Gets bounding box of the non-black pixels
    if bbox:
        img = img.crop(bbox)

    # 5. Padding & Resizing: Add a tiny 15% border (standard for EMNIST), then resize to 32x32
    padding = max(img.size) // 6
    img = ImageOps.expand(img, border=padding, fill=0)
    img = img.resize((32, 32), Image.Resampling.BILINEAR)

    # 6. Fix EMNIST Rotation Quirk
    if fix_emnist_rotation:
        img = img.rotate(-90, expand=True)
        img = ImageOps.mirror(img)

    # 7. Convert to tensor and apply your training normalization
    transform_pipeline = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    return transform_pipeline(img).unsqueeze(0)

# --- Full EMNIST Decoder Dictionary ---
emnist_decoder = {}
for i in range(10): emnist_decoder[i] = str(i)             # 0-9
for i in range(26): emnist_decoder[i + 10] = chr(i + 65)   # A-Z
for i in range(26): emnist_decoder[i + 36] = chr(i + 97)   # a-z

# --- 3. Evaluation Loop ---
def evaluate_custom_dataset(model, folder_path, device):
    model.eval()

    results = {
        'total': {'strict_correct': 0, 'case_insensitive_correct': 0, 'count': 0},
        'messy': {'strict_correct': 0, 'case_insensitive_correct': 0, 'count': 0},
        'print': {'strict_correct': 0, 'case_insensitive_correct': 0, 'count': 0}
    }

    print(f"Evaluating images in: {folder_path}...\n")

    with torch.no_grad():
        for filename in os.listdir(folder_path):
            if not filename.endswith(('.png', '.jpg', '.jpeg')):
                continue

            parts = filename.replace('.png', '').replace('.jpg', '').split('_')
            if len(parts) != 3:
                continue

            writing_type = parts[1]
            custom_class = int(parts[2])

            true_char_lower = custom_to_char[custom_class]
            true_emnist_label = char_to_emnist_byclass[true_char_lower]

            img_path = os.path.join(folder_path, filename)
            img_tensor = preprocess_custom_image(img_path).to(device)

            outputs = model(img_tensor)
            _, predicted_label = torch.max(outputs.data, 1)
            predicted_label = predicted_label.item()

            # Use our new decoder to get the actual predicted letter!
            pred_char = emnist_decoder[predicted_label]

            # --- Check Accuracy ---
            # Strict: Must match exactly (e.g., lowercase 'w' == lowercase 'w')
            strict_match = (predicted_label == true_emnist_label)

            # Case-Insensitive: Matches if letters are the same, regardless of case
            case_match = (pred_char.lower() == true_char_lower.lower())

            # Update stats
            results['total']['count'] += 1
            results[writing_type]['count'] += 1

            if strict_match:
                results['total']['strict_correct'] += 1
                results[writing_type]['strict_correct'] += 1
            if case_match:
                results['total']['case_insensitive_correct'] += 1
                results[writing_type]['case_insensitive_correct'] += 1

            # Print failures (only print if it failed the case-insensitive check)
            if not case_match:
                print(f"True Mistake on {filename}: True={true_char_lower}, Predicted={pred_char}")

    # --- Print Summary ---
    print("\n--- Custom Dataset Results ---")
    for category in ['total', 'messy', 'print']:
        count = results[category]['count']
        if count > 0:
            strict_acc = results[category]['strict_correct'] / count * 100
            case_acc = results[category]['case_insensitive_correct'] / count * 100
            print(f"{category.capitalize()} - Strict Accuracy: {strict_acc:.2f}% | Case-Insensitive Accuracy: {case_acc:.2f}%")

# --- 4. Execution ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_classes = 62
model = LeNet5(num_classes).to(device)

# 3. Load the Weights
model_path = "/content/drive/MyDrive/cs320/final project/models/lenet5_rbf_byclass_final.pth"

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Successfully loaded model weights from {model_path}")

    folder_dir = "/content/drive/MyDrive/cs320/final project/handwriting_data"

    if os.path.exists(folder_dir):
        evaluate_custom_dataset(model, folder_dir, device)
    else:
        print(f"Error: Could not find image folder at {folder_dir}. Check the path.")
else:
    print(f"Error: Could not find model file at {model_path}. Check the path.")

Using device: cuda
Successfully loaded model weights from /content/drive/MyDrive/cs320/final project/models/lenet5_rbf_byclass_final.pth
Evaluating images in: /content/drive/MyDrive/cs320/final project/handwriting_data...

True Mistake on sample5_messy_1.png: True=q, Predicted=m
True Mistake on sample4_messy_9.png: True=h, Predicted=m
True Mistake on sample4_print_22.png: True=r, Predicted=m
True Mistake on sample5_print_10.png: True=g, Predicted=m
True Mistake on sample3_messy_2.png: True=l, Predicted=m
True Mistake on sample4_messy_3.png: True=p, Predicted=j
True Mistake on sample4_messy_19.png: True=i, Predicted=m
True Mistake on sample5_print_19.png: True=i, Predicted=m
True Mistake on sample5_messy_20.png: True=s, Predicted=m
True Mistake on sample5_messy_22.png: True=r, Predicted=m
True Mistake on sample4_messy_1.png: True=q, Predicted=m
True Mistake on sample5_messy_8.png: True=n, Predicted=m
True Mistake on sample5_messy_17.png: True=j, Predicted=m
True Mistake on sample4_print